In [ ]:
!pip install -U langchain-ollama langchain-openai

In [55]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [56]:
!ls /content/drive/MyDrive/review_dataset

w_review_train.csv  w_review_train.parquet


In [57]:
import duckdb

con = duckdb.connect(database=':memory:', read_only=False)

In [58]:
df = con.execute("SELECT * FROM read_csv_auto('/content/drive/MyDrive/review_dataset/w_review_train.csv', parallel=false, encoding='UTF-8', ignore_errors=true)").fetchdf()

df.head()

,column0,column1
0,ร้านอาหารใหญ่มากกกกกกก \nเลี้ยวเข้ามาเจอห้องน้...,3
1,อาหารที่นี่เป็นอาหารจีนแคะที่หากินยากในบ้านเรา...,4
2,ปอเปี๊ยะสด ทุกวันนี้รู้สึกว่าหากินยาก (ร้านที่...,3
3,รัานคัพเค้กในเมืองไทยมีไม่มาก หลายๆคนอาจจะสงสั...,5
4,อร่อย!!! เดินผ่านDigital gatewayทุกวัน ไม่ยักร...,5


In [59]:
df.shape

(40000, 2)

In [60]:
con.execute("COPY df TO '/content/drive/MyDrive/review_dataset/w_review_train.parquet' (FORMAT PARQUET)")

In [61]:
parquet_avg_rating_direct = con.execute("SELECT AVG(column1) FROM '/content/drive/MyDrive/review_dataset/w_review_train.parquet' WHERE column0 LIKE '%อร่อย%'").fetchone()[0]
parquet_avg_rating_direct

3.8313027179006562

In [62]:
parquet_file_path = '/content/drive/MyDrive/review_dataset/w_review_train.parquet'

In [63]:
parquet_avg_rating_direct = con.execute(f"SELECT AVG(column1) FROM '{parquet_file_path}' WHERE column0 LIKE '%อร่อย%'").fetchone()[0]
parquet_avg_rating_direct

3.8313027179006562

In [64]:
coffee_keywords = [
    'กาแฟ', 'คาเฟ่', 'coffee', 'cafe', 'เอสเปรสโซ่', 'ลาเต้',
    'มอคค่า', 'คาปูชิโน่', 'เฟรนช์เพรส', 'cold brew', 'drip',
    'espresso', 'latte', 'americano', 'macchiato', 'brew',
    'specialty', 'slow bar', 'ร้านกาแฟ', 'ร้านคาเฟ่', 'กาแฟสด',
    'เมนูกาแฟ', 'กาแฟนม', 'กาแฟดำ', 'ร้านบาริสต้า', 'barista', 'coffee shop'
]

where_clause = " OR ".join([f"column0 LIKE '%{keyword}%'" for keyword in coffee_keywords])

In [65]:
%%time
coffee_reviews_df = con.execute(f"SELECT * FROM '{parquet_file_path}' WHERE {where_clause}").fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

CPU times: user 4.9 s, sys: 48.9 ms, total: 4.94 s
Wall time: 2.61 s


In [66]:
csv_file_path = '/content/drive/MyDrive/review_dataset/w_review_train.csv'

In [67]:
%%time
coffee_reviews_df_csv = con.execute(f"SELECT * FROM read_csv_auto('{csv_file_path}', parallel=false, encoding='UTF-8', ignore_errors=true) WHERE {where_clause}").fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

CPU times: user 2.22 s, sys: 57.1 ms, total: 2.28 s
Wall time: 2.31 s


In [68]:
coffee_reviews_df.head()

,column0,column1
0,ร้านอาหารใหญ่มากกกกกกก \nเลี้ยวเข้ามาเจอห้องน้...,3
1,วันนี้ได้มีโอกาสไปนั่งซดกาแฟที่ร้านวาวี แถวๆอา...,4
2,สารภาพว่าไม่เคยคิดจะไปต่อคิวซื้อมากินเองครับ บ...,3
3,ร้านอาหารญี่ปุ่นร้านนี้ ใจจริงไม่อยากแนะนำเลยค...,5
4,เดือนแรกที่เค้าต่อคิวกัน 2 - 3 ชั่วโมง เราก็ว่...,5


In [69]:
coffee_reviews_df.shape

(4981, 2)

In [70]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [71]:
!nohup ollama serve &

nohup: appending output to 'nohup.out'


In [72]:
!ollama pull gemma3:270m

In [73]:
!ollama list

NAME           ID              SIZE      MODIFIED               
gemma3:270m    e7d36fb2c3b3    291 MB    Less than a second ago    


In [74]:
!ollama run gemma3:270m "สวัสดี"

สวัสดีค่ะ ยินดีที่ได้รู้จักค่ะ 😊




In [75]:
!curl -s http://127.0.0.1:11434/api/tags

{"models":[{"name":"gemma3:270m","model":"gemma3:270m","modified_at":"2025-10-19T10:49:09.215116919Z","size":291554930,"digest":"e7d36fb2c3b3293cfe56d55889867a064b3a2b22e98335f2e6e8a387e081d6be","details":{"parent_model":"","format":"gguf","family":"gemma3","families":["gemma3"],"parameter_size":"268.10M","quantization_level":"Q8_0"}}]}

In [76]:
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [77]:
llm = OllamaLLM(model="gemma3:270m")

In [78]:
response = llm.invoke("สวัสดี")
print(response)

สวัสดีค่ะ ยินดีที่ได้รู้จักค่ะ 😊



In [79]:
template = """
คุณเป็นครูสอน Python
คำถาม : {question}
"""

In [80]:
prompt = PromptTemplate(
    input_variables=["question"],
    template=template
)

In [81]:
formatted_prompt = prompt.format(question="เขียนโค้ดหาผลรวมของตัวเลข 1 ถึง 10")
print(formatted_prompt)


คุณเป็นครูสอน Python
คำถาม : เขียนโค้ดหาผลรวมของตัวเลข 1 ถึง 10



In [82]:
chain = prompt | llm

In [83]:
response = chain.invoke({"question": "เขียนโค้ดหาผลรวมของตัวเลข 1 ถึง 10"})

In [84]:
response

'```python\ndef find_sum(numbers):\n    """\n    หาผลรวมของตัวเลขในชุดข้อมูล\n    """\n    try:\n        total = 0\n        for number in numbers:\n            total += number\n        return total\n    except Exception as e:\n        print(f"ข้อผิดพลาด: {e}")\n        return 0  # หรือค่าที่กำหนดให้เป็น 0\n    else:\n        return 0  # หรือค่าที่กำหนดให้เป็น 0\n```\n\n**คำอธิบาย:**\n\n1.  **`def find_sum(numbers):`**:  กำหนดชื่อฟังก์ชัน `find_sum` ที่จะทำงาน\n2.  **`try...except`**:  เป็นการจัดการกับข้อผิดพลาดที่อาจเกิดขึ้นเมื่อทำการค้นหาผลรวม\n3.  **`total = 0`**:  เริ่มต้นค่า `total` เป็น 0\n4.  **`for number in numbers:`**:  วนรอบผ่านแต่ละตัวเลขในชุดข้อมูล `numbers`\n5.  **`total += number`**:  เพิ่มค่าของ `number` เข้าไปในค่า `total`\n6.  **`return total`**:  คืนค่าค่า `total`\n7.  **`except Exception as e:`**:  หากเกิดข้อผิดพลาด (เช่น การเรียกใช้ API ไม่สำเร็จ หรือเกิดข้อผิดพลาดอื่นๆ) จะทำการตรวจสอบข้อผิดพลาดและส่งข้อความแจ้งเตือน (Error message) ให้ผู้ใช้\n\n**การใช้งานโค้ด:**\n

In [85]:
from IPython.display import display, Markdown

In [86]:
display(Markdown(response))

```python
def find_sum(numbers):
    """
    หาผลรวมของตัวเลขในชุดข้อมูล
    """
    try:
        total = 0
        for number in numbers:
            total += number
        return total
    except Exception as e:
        print(f"ข้อผิดพลาด: {e}")
        return 0  # หรือค่าที่กำหนดให้เป็น 0
    else:
        return 0  # หรือค่าที่กำหนดให้เป็น 0
```

**คำอธิบาย:**

1.  **`def find_sum(numbers):`**:  กำหนดชื่อฟังก์ชัน `find_sum` ที่จะทำงาน
2.  **`try...except`**:  เป็นการจัดการกับข้อผิดพลาดที่อาจเกิดขึ้นเมื่อทำการค้นหาผลรวม
3.  **`total = 0`**:  เริ่มต้นค่า `total` เป็น 0
4.  **`for number in numbers:`**:  วนรอบผ่านแต่ละตัวเลขในชุดข้อมูล `numbers`
5.  **`total += number`**:  เพิ่มค่าของ `number` เข้าไปในค่า `total`
6.  **`return total`**:  คืนค่าค่า `total`
7.  **`except Exception as e:`**:  หากเกิดข้อผิดพลาด (เช่น การเรียกใช้ API ไม่สำเร็จ หรือเกิดข้อผิดพลาดอื่นๆ) จะทำการตรวจสอบข้อผิดพลาดและส่งข้อความแจ้งเตือน (Error message) ให้ผู้ใช้

**การใช้งานโค้ด:**

```python
numbers = [1, 2, 3, 4, 5]
sum = find_sum(numbers)
print(sum)  # Output: 15
```

**ข้อดีของโค้ดนี้:**

*   **ง่ายและเข้าใจ:**  โค้ดนี้เป็นภาษา Python ที่เข้าใจง่ายและสามารถนำไปใช้งานได้โดยตรง
*   **Efficient:**  การใช้ `for` loop เป็นวิธีที่ประหยังในการคำนวณผลรวมของตัวเลข
*   **สามารถปรับแต่งได้:**  คุณสามารถปรับแต่งค่า `numbers` เพื่อให้โค้ดทำงานได้ดีที่สุด
*   **จัดการกับข้อผิดพลาด:**  การจัดการข้อผิดพลาดช่วยให้โค้ดสามารถทำงานได้อย่างมีประสิทธิภาพ

**คำแนะนำเพิ่มเติม:**

*   **ตัวอย่าง:**  คุณสามารถเพิ่มตัวอย่างโค้ดที่แสดงให้เห็นการใช้งานโค้ดนี้ได้
*   **การจัดการกับข้อผิดพลาด:**  คุณสามารถกำหนดการจัดการข้อผิดพลาดสำหรับโค้ดนี้ได้ เช่น การแสดงผลแจ้งเตือน หรือการส่งข้อความแจ้งเตือน
*   **การจัดการกับค่า `0`:**  คุณสามารถกำหนดค่า `0` ให้เป็นค่าที่กำหนดเองให้กับ `total` เพื่อให้โค้ดทำงานได้อย่างถูกต้อง



In [87]:
prompt1 = PromptTemplate(
    template="แปลโจทย์เกี่ยวกับ {question}"
)

chain1 = prompt1 | llm

In [88]:
prompt2 = PromptTemplate(
    template="ทำความเข้าใจโจทย์ที่แปลแล้วต่อไปนี้ {translate}"
)

chain2 = prompt2 | llm

In [89]:
prompt3 = PromptTemplate(
    template="แก้โจทย์ตามที่เข้าใจต่อไปนี้ {understand}"
)

chain3 = prompt3 | llm

In [90]:
prompt4 = PromptTemplate(
    template="อธิบายวิธิีทำจากผลลัทธ์ต่อไปนี้ {solve}"
)

chain4 = prompt4 | llm

In [91]:
parser_chain = StrOutputParser()

In [92]:
full_chain = chain1 | chain2 | chain3 | chain4 | parser_chain

In [93]:
result = full_chain.invoke({"question":"2x + 5 = 15"})

In [94]:
result

'ผลลัทธ์เป็นเครื่องมือสำคัญที่ช่วยให้เราเข้าใจและจัดการกับปัญหาต่างๆ ได้อย่างมีประสิทธิภาพ โดยมีหลักการสำคัญดังนี้ครับ'

In [95]:
import os
from langchain_openai import ChatOpenAI

In [96]:
OPENAI_API_KEY = "xxx"

os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

In [97]:
llm2 = ChatOpenAI(model="gpt-4o-mini")

In [98]:
chain1 = prompt1 | llm2
chain2 = prompt2 | llm2
chain3 = prompt3 | llm2
chain4 = prompt4 | llm2

full_chain = chain1 | chain2 | chain3 | chain4 | parser_chain

In [99]:
formatted_result = result.replace('\\[', '$$') \
                         .replace('\\]', '$$') \
                         .replace('\\(', '$') \
                         .replace('\\)', '$')

In [100]:
display(Markdown(formatted_result))

ผลลัทธ์เป็นเครื่องมือสำคัญที่ช่วยให้เราเข้าใจและจัดการกับปัญหาต่างๆ ได้อย่างมีประสิทธิภาพ โดยมีหลักการสำคัญดังนี้ครับ

In [101]:
prompt_template = """
[1. Role/Context บอก LLM ว่าควรทำตัวเป็นอะไร]
คุณเป็น AI ที่เชี่ยวชาญในการวิเคราะห์รีวิวภาษาไทย

[2. Task Description บอกงานที่ต้องทำ]
จาก Review ต่อไปนี้ ให้ Extract 3 keywords
ที่เป็นตัวแทนของ Review นี้ได้ดีที่สุด

[3. Input Data ข้อมูลที่ต้องประมวลผล]
Review: {review}

[4. Output Format บอกรูปแบบของคำตอบ]
ตอบในรูปแบบ JSON โดยมีคีย์เป็น 'keywords'
เช่น {{"keywords": "กาแฟ, บรรยากาศ, ราคา"}

[5. Constraints ข้อจำกัดหรือเงื่อนไข]
- ต้องเป็นคำภาษาไทย
- คั่นด้วย comma
- หากไม่พบให้ตอบ 'ไม่ระบุ'

[6. Examples (optional) ตัวอย่าง input/output]
ตัวอย่าง
Input: "กาแฟอร่อย บรรยากาศดี ราคาแพง"
Output: {"keywords": "กาแฟ, บรรยากาศ, ราคา"}
"""

In [102]:
prompt = """
Extract keywords from this review
{review}
"""

# Pros Simple, fast
# Cons อาจได้ผลไม่ดีกับ Tasks ซับซ้อน

In [103]:
prompt = """
Extract keywords from reviews.

Example 1
Review: "กาแฟอร่อย บริการดี"
Keywords: กาแฟ, บริการ

Example 2
Review: "ราคาแพง แต่คุ้มค่า"
Keywords: ราคา, คุณภาพ

Now extract from this review
Review: {review}
Keywords:
"""

# Pros ได้ผลดีกว่า Zero-shot
# Cons Prompt ใช้ Tokens มากขึ้น

In [104]:
prompt = """
Review: {review}

ขั้นตอนการ Extract keywords
1. อ่านรีวิวและเข้าใจความหมาย
2. ระบุหัวข้อหลักที่กล่าวถึง
3. เลือก 3 keywords ที่สำคัญที่สุด
4. ตอบในรูปแบบ JSON: {{"keywords": "คำ1, คำ2, คำ3"}}

ให้ทำทีละขั้นตอน
"""

# Pros ได้ผลดีกับ Reasoning tasks
# Cons ใช้ Tokens มากกว่า ช้ากว่า

In [105]:
keyword_prompt = PromptTemplate(
template = """
จาก Review ต่อไปนี้
Extract 3 keywords ที่เป็นตัวแทนของ Review นี้

Review: {review}

ขั้นตอนการ Extract keywords
1. อ่านรีวิวและเข้าใจความหมาย
2. ระบุหัวข้อหลักที่กล่าวถึง
3. เลือก 3 keywords ที่สำคัญที่สุด
4. ตอบในรูปแบบ JSON โดยมี key คือ keyword เช่น :{{"keywords": "คำ1, คำ2, คำ3"}}

ให้ทำทีละขั้นตอน
""")

In [106]:
sample_reviews = coffee_reviews_df['column0'].head(2)
sample_reviews

,column0
0,ร้านอาหารใหญ่มากกกกกกก \nเลี้ยวเข้ามาเจอห้องน้...
1,วันนี้ได้มีโอกาสไปนั่งซดกาแฟที่ร้านวาวี แถวๆอา...


In [107]:
sample_reviews.to_csv('/content/sample_reviews.csv', index=False)

In [108]:
keyword_extract = keyword_prompt | llm | StrOutputParser()

In [109]:
%%time
results = []
for review in sample_reviews:
    text = keyword_extract.invoke({"review": review})
    cleaned = (text).replace("**", "").strip()
    results.append({"review": review, "text": cleaned})

CPU times: user 37.7 ms, sys: 3.58 ms, total: 41.3 ms
Wall time: 5.44 s


In [110]:
for result in results:
    print(f"Review {result['review'][:100]}...")
    print(f"text {result['text']}\n")

Review ร้านอาหารใหญ่มากกกกกกก 
เลี้ยวเข้ามาเจอห้องน้ำก่อนเลย เออแปลกดี 
ห้องทานหลักๆอยู่ชั้น 2 มีกาแฟ น้ำผึ...
text ```json
{
  "keywords": [
    "ร้านอาหารใหญ่",
    "ห้องน้ำ",
    "กาแฟ",
    "น้ำผึ้ง",
    "ความอร่อย",
    "ราคา",
    "ไม่ประทับใจ",
    "เกินไป",
    "รับไม่ไหว",
    "ว"
  ]
}
```

Review วันนี้ได้มีโอกาสไปนั่งซดกาแฟที่ร้านวาวี แถวๆอารีย์

ก็ได้ยินมาบ่อยๆละ จากหลายที่ แต่ที่บ่อยสุดก็จาก ...
text ```json
{
  "keywords": [
    "ลาเต้",
    "ร้อน",
    "แก้วนึง",
    "ร้อน",
    "มี",
    "ซากอารยธรรม",
    "เมพพพ",
    "เมพพพพ",
    "ซากอารยธรรม",
    "ซากอารยธรรม"
  ]
}
```



In [111]:
keyword_extract = keyword_prompt | llm2 | StrOutputParser()

In [112]:
for result in results:
    print(f"Review {result['review'][:100]}...")
    print(f"text {result['text']}\n")

Review ร้านอาหารใหญ่มากกกกกกก 
เลี้ยวเข้ามาเจอห้องน้ำก่อนเลย เออแปลกดี 
ห้องทานหลักๆอยู่ชั้น 2 มีกาแฟ น้ำผึ...
text ```json
{
  "keywords": [
    "ร้านอาหารใหญ่",
    "ห้องน้ำ",
    "กาแฟ",
    "น้ำผึ้ง",
    "ความอร่อย",
    "ราคา",
    "ไม่ประทับใจ",
    "เกินไป",
    "รับไม่ไหว",
    "ว"
  ]
}
```

Review วันนี้ได้มีโอกาสไปนั่งซดกาแฟที่ร้านวาวี แถวๆอารีย์

ก็ได้ยินมาบ่อยๆละ จากหลายที่ แต่ที่บ่อยสุดก็จาก ...
text ```json
{
  "keywords": [
    "ลาเต้",
    "ร้อน",
    "แก้วนึง",
    "ร้อน",
    "มี",
    "ซากอารยธรรม",
    "เมพพพ",
    "เมพพพพ",
    "ซากอารยธรรม",
    "ซากอารยธรรม"
  ]
}
```



In [113]:
!ollama pull scb10x/typhoon2.1-gemma3-4b

In [114]:
!ollama list

NAME                                  ID              SIZE      MODIFIED               
scb10x/typhoon2.1-gemma3-4b:latest    8cfab9097c9d    2.6 GB    Less than a second ago    
gemma3:270m                           e7d36fb2c3b3    291 MB    About a minute ago        


In [115]:
llm3 = OllamaLLM(model="gemma3:270m")

In [116]:
keyword_extract = keyword_prompt | llm3 | StrOutputParser()

In [117]:
%%time
results = []
for review in sample_reviews:
    text = keyword_extract.invoke({"review": review})
    cleaned = (text).replace("**", "").strip()
    results.append({"review": review, "text": cleaned})

CPU times: user 309 ms, sys: 29.6 ms, total: 339 ms
Wall time: 16.5 s


In [118]:
for result in results:
    print(f"Review {result['review'][:100]}...")
    print(f"text {result['text']}\n")

Review ร้านอาหารใหญ่มากกกกกกก 
เลี้ยวเข้ามาเจอห้องน้ำก่อนเลย เออแปลกดี 
ห้องทานหลักๆอยู่ชั้น 2 มีกาแฟ น้ำผึ...
text ```json
{
  "keywords": ["ร้านอาหาร", "ห้องน้ำ", "กาแฟ", "น้ำผึ้ง", "น้ำผึ้ง", "ลาบไข่ต้ม", "ไข่มัน", "ขนมไทย", "ขนมหวาน", "ฟรี", "ราคา", "ราคาเกินไป", "ราคาสูง", "ราคาสินค้า", "ร้านอาหาร", "ร้านอาหารใหญ่", "ร้านอาหารราคาถูก", "ร้านอาหารราคาไม่แพง", "ร้านอาหารดี", "ร้านอาหารดีมาก", "ร้านอาหารดีมากเลย", "ร้านอาหารดีมากเลย", "ร้านอาหารดีมากเลย", "ร้านอาหารดีมากเลย", "ร้านอาหารดีมากเลย", "ร้านอาหารดีมากเลย", "ร้านอาหารดีมากเลย", "ร้านอาหารดีมากเลย", "ร้านอาหารดีมากเลย", "ร้านอาหารดีมากเลย", "ร้านอาหารดีมากเลย", "ร้านอาหารดีมากเลย", "ร้านอาหารดีมากเลย", "ร้านอาหารดีมากเลย", "ร้านอาหารดีมากเลย", "ร้านอาหารดีมากเลย", "ร้านอาหารดีมากเลย", "ร้านอาหารดีมากเลย", "ร้านอาหารดีมากเลย", "ร้านอาหารดีมากเลย", "ร้านอาหารดีมากเลย", "ร้านอาหารดีมากเลย", "ร้านอาหารดีมากเลย", "ร้านอาหารดีมากเลย", "ร้านอาหารดีมากเลย", "ร้านอาหารดีมากเลย", "ร้านอาหารดีมากเลย", "ร้านอาหารดีมากเลย", "ร้านอาหารดีม

In [119]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser

class ReviewKeywords(BaseModel):
    keywords: str = Field(
        ...,  # Required field
        description="3 keywords separated by comma",
        examples=["กาแฟ, บรรยากาศ, ราคา"]
    )

class KeywordsExtraction(BaseModel):
    extracted_qualities: ReviewKeywords

In [120]:
keywords_parser = JsonOutputParser(pydantic_object=KeywordsExtraction)

In [121]:
keyword_extract = keyword_prompt | llm3 | keywords_parser

In [122]:
%%time
results = []
for review in sample_reviews:
    text = keyword_extract.invoke({"review": review})
    results.append({"review": review, "text": text})

CPU times: user 59.9 ms, sys: 2.41 ms, total: 62.3 ms
Wall time: 5.12 s


In [123]:
results[0]['text']['keywords']

['ร้านอาหารใหญ่',
 'ห้องน้ำ',
 'กาแฟ',
 'น้ำผึ้ง',
 'น้ำผึ้งมาราด',
 'แพงเวอร์',
 'อย่าสั่งเลย',
 'ลาบไข่ต้ม',
 'ไข่มันคาว',
 'ประทับใจ',
 'เบิ้ล',
 'ขนมไทย']

In [124]:
results[1]['text']['keywords']

['ลาเต้',
 'รสชาติ',
 'กลิ่น',
 'มวยรุ่นเดียวกะ Starbuck',
 'เมพพพพ',
 'Coco',
 'เครื่องดื่ม',
 'กาแฟ',
 'ร้านวาวี',
 'อารีย์',
 'ซัมซวย',
 'Omnia i900',
 'สีภาพถึงได้ไม่ค่อยสวยเท่าไหร',
 'การถ่ายรูป',
 '55+',
 'ร้าน',
 'Villa',
 'BS อารีย์']

In [125]:
llm3 = OllamaLLM(
    model="scb10x/typhoon2.1-gemma3-4b",
    temperature=0.05
)

keyword_extract = keyword_prompt | llm3 | keywords_parser

In [126]:
%%time
results = []
for review in sample_reviews:
    text = keyword_extract.invoke({"review": review})
    results.append({"review": review, "text": text})

CPU times: user 171 ms, sys: 11 ms, total: 182 ms
Wall time: 19.5 s


In [127]:
results[0]['text']['keywords'], results[1]['text']['keywords']

(['ราคา', 'รสชาติ', 'ขนาด'], ['กาแฟ', 'บรรยากาศ', 'วาวี'])

In [128]:
keyword_prompt = PromptTemplate(
    template="""จาก Review ต่อไปนี้
    Extract keywords 3 keywords เท่านั้น ที่เป็นตัวแทนของ Review นี้ได้ดีที่สุด

    Review: {review}

    ตอบในรูปแบบ JSON โดยมีคีย์เป็น 'keywords'
    และค่าเป็นข้อความที่ดึงได้คั่นด้วย comma
    เช่น {{"keywords": "กาแฟ, บรรยากาศ, ราคา"}}

    หากไม่พบข้อมูลให้ตอบ 'ไม่ระบุ'
    """
)

In [129]:
keyword_extract = keyword_prompt | llm3 | keywords_parser

In [130]:
inputs = [{"review": r} for r in sample_reviews]

In [131]:
%%time
results = keyword_extract.batch(inputs)

CPU times: user 23.6 ms, sys: 118 µs, total: 23.7 ms
Wall time: 2.61 s


In [132]:
results

[{'keywords': 'ราคา, บรรยากาศ, ไข่ต้ม'},
 {'keywords': 'กาแฟ, บรรยากาศ, รสชาติ'}]

In [133]:
sample_reviews_1000 = coffee_reviews_df['column0'].head(100).tolist()
inputs_1000 = [{"review": r} for r in sample_reviews_1000]

BATCH_SIZE = 8

results_1000 = []

In [134]:
from tqdm import tqdm

In [135]:
for i in tqdm(range(0, len(inputs_1000), BATCH_SIZE), desc="Extracting keywords"):
    batch = inputs_1000[i:i+BATCH_SIZE]
    outputs = keyword_extract.batch(batch)
    for inp, out in zip(batch, outputs):
        results_1000.append({"review": inp["review"], "text": out})

Extracting keywords: 100%|██████████| 13/13 [02:15<00:00, 10.41s/it]


In [136]:
for result in results_1000[0:10]:
    print(result['text']['keywords'])

ราคา, บรรยากาศ, ไข่ต้ม
กาแฟ, บรรยากาศ, รสชาติ
โดนัท, รสชาติ, ความหวาน
วาซาบิ, ปลาซาบะ, ข้าวหน้าปลา
โดนัท, Original Glazed, คิว
วังพญาไท, นรสิงห์, คาเฟ่
อาหารทะเล, ราคา, บรรยากาศ
กาแฟ, เค้ก, พนักงาน
ฮันนี่ โทสท์, ราคา, รสชาติ
คาเฟ่, วานิลลา, คาโบนาร่า


In [137]:
  results_1000[0]

{'review': 'ร้านอาหารใหญ่มากกกกกกก \nเลี้ยวเข้ามาเจอห้องน้ำก่อนเลย เออแปลกดี \nห้องทานหลักๆอยู่ชั้น 2 มีกาแฟ น้ำผึ้ง ซึ่งก็แค่เอาน้ำผึ้งมาราด แพงเวอร์ อย่าสั่งเลย \nลาบไข่ต้ม ไข่มันคาวอะ เลยไม่ประทับใจเท่าไหร่\nทอดมันหัวปลีกรอบอร่อยต้องเบิ้ล \nพะแนงห่อไข่อร่อยดี เห้ยแต่ราคา 150บาทมันเกินไปนะ รับไม่ไหวว\nเลิกกินแล้วมีขนมหวานให้กินฟรีเล็กน้อย )ขนมไทย) \n\nคงไม่ไปซ้ำ แพงเกิน ',
 'text': {'keywords': 'ราคา, บรรยากาศ, ไข่ต้ม'}}

In [138]:
context_prompt = PromptTemplate(
    template="""จาก Review ต่อไปนี้
    Extract context ใน Review ที่เป็นตัวแทนที่บรรยายคำว่า {keyword} ได้ดีที่สุด

    Review: {review}

    ตอบผลลัพธ์ในรูปแบบ JSON โดยมีคีย์เป็น 'context'
    และค่าเป็นข้อความที่ดึงได้จาก Review

    ตัวอย่าง
    keyword คือ "กาแฟ"
    context: "กาแฟที่นี่อร่อยมาก หอมกลิ่นกาแฟคั่วสด"

    หากไม่พบข้อมูลให้ตอบ 'ไม่ระบุ'
    """
)

In [139]:
sentiment_prompt = PromptTemplate(
    template="""จากข้อความต่อไปนี้ ให้วิเคราะห์ว่าเป็น sentiment แบบไหน
    เลือกจากตัวเลือกเหล่านี้เท่านั้น: "positive", "negative", "neutral"

    ข้อความ: {message}

    ตอบในรูปแบบ JSON โดยมีคีย์เป็น 'sentiment'
    และค่าเป็นหนึ่งใน list ต่อไปนี้ [positive, negative, neutral]

    คำแนะนำ
    - positive คือ ข้อความที่แสดงความพอใจ ชอบ ดี
    - negative คือ ข้อความที่แสดงความไม่พอใจ ไม่ชอบ แย่
    - neutral คือ ข้อความที่ไม่แสดงความรู้สึกชัดเจน เป็นกลาง

    # ตัวอย่าง
    # ข้อความ "อร่อยมาก" → {{"sentiment": "positive"}}
    # ข้อความ "แพงไป" → {{"sentiment": "negative"}}
    # ข้อความ "ปกติ" → {{"sentiment": "neutral"}}
    """
)


In [140]:
class ContextKeyword(BaseModel):
    context: str = Field(..., description="context that best represent this keyword")

class ContexExtraction(BaseModel):
    extracted: ContextKeyword

context_parser = JsonOutputParser(pydantic_object=ContexExtraction)

In [141]:
context_extract = context_prompt | llm3 | context_parser

In [142]:
class Sentiment(BaseModel):
    sentiment: str = Field(..., description="sentiment of review")

class SentimentExtraction(BaseModel):
    extracted: Sentiment


sentiment_parser = JsonOutputParser(pydantic_object=SentimentExtraction)

In [143]:
sentiment_extract = sentiment_prompt | llm3 | sentiment_parser

In [144]:
keywords = results_1000[0]['text']['keywords']


In [145]:
context_inputs = []

for review_data in tqdm(results_1000, desc="Preparing context inputs"):
    review = review_data['review']
    keywords = review_data['text']['keywords']
    for keyword in keywords:
        context_inputs.append({"review": review, "keyword": keyword.strip()})

Preparing context inputs: 100%|██████████| 100/100 [00:00<00:00, 58925.32it/s]


In [146]:
len(context_inputs)

2521

In [147]:
context_results = []
for i in tqdm(range(len(results_1000)), desc="Extracting contexts"):
   result = context_extract.batch(context_inputs[i*3:(i*3)+3])
   contexts_list = [d['context'] for d in result]
   results_1000[i]['contexts'] = contexts_list

Extracting contexts: 100%|██████████| 100/100 [09:06<00:00,  5.47s/it]


In [148]:
results_1000[:2]

[{'review': 'ร้านอาหารใหญ่มากกกกกกก \nเลี้ยวเข้ามาเจอห้องน้ำก่อนเลย เออแปลกดี \nห้องทานหลักๆอยู่ชั้น 2 มีกาแฟ น้ำผึ้ง ซึ่งก็แค่เอาน้ำผึ้งมาราด แพงเวอร์ อย่าสั่งเลย \nลาบไข่ต้ม ไข่มันคาวอะ เลยไม่ประทับใจเท่าไหร่\nทอดมันหัวปลีกรอบอร่อยต้องเบิ้ล \nพะแนงห่อไข่อร่อยดี เห้ยแต่ราคา 150บาทมันเกินไปนะ รับไม่ไหวว\nเลิกกินแล้วมีขนมหวานให้กินฟรีเล็กน้อย )ขนมไทย) \n\nคงไม่ไปซ้ำ แพงเกิน ',
  'text': {'keywords': 'ราคา, บรรยากาศ, ไข่ต้ม'},
  'contexts': ['ร้านอาหารใหญ่มากกกกกกก',
   'ร้านอาหารใหญ่มากกกกกกก เลี้ยวเข้ามาเจอห้องน้ำก่อนเลย เออแปลกดี',
   'ห้องทานอาหารอยู่ชั้น 2 มีกาแฟ น้ำผึ้ง ซึ่งก็แค่เอาน้ำผึ้งมาราด แพงเวอร์ อย่าสั่งเลย']},
 {'review': 'วันนี้ได้มีโอกาสไปนั่งซดกาแฟที่ร้านวาวี แถวๆอารีย์\n\nก็ได้ยินมาบ่อยๆละ จากหลายที่ แต่ที่บ่อยสุดก็จาก Twitter ว่ากาแฟที่นี่อร่อยมากกกกกกก\nเรียกว่าระดับแฟนๆ Starbuck ยังต้องเหลียวมามอง ก็ดองมานานจนถึงวันนี้โอกาสเหมาะ ไป\npresent งานที่ตึก Software Park เสร็จก็เลยมาหาไรกินแถวนี้พอดี\nจัดไป.....\n\n\nบรรยากาศรอบๆร้านก็แต่งได้น่านั่ง อารมณ์ประมาณว่าอยู่ในสว

In [149]:
sentiment_inputs = []

for review_data in tqdm(results_1000, desc="Preparing sentiment inputs"):
    contexts = review_data['contexts']
    for context in contexts:
        sentiment_inputs.append({"message": context})

Preparing sentiment inputs: 100%|██████████| 100/100 [00:00<00:00, 266813.23it/s]


In [150]:
len(sentiment_inputs)

300

In [151]:
sentiment_inputs[:6]

[{'message': 'ร้านอาหารใหญ่มากกกกกกก'},
 {'message': 'ร้านอาหารใหญ่มากกกกกกก เลี้ยวเข้ามาเจอห้องน้ำก่อนเลย เออแปลกดี'},
 {'message': 'ห้องทานอาหารอยู่ชั้น 2 มีกาแฟ น้ำผึ้ง ซึ่งก็แค่เอาน้ำผึ้งมาราด แพงเวอร์ อย่าสั่งเลย'},
 {'message': 'ร้านอาหารใหญ่มากกกกกกก เลี้ยวเข้ามาเจอห้องน้ำก่อนเลย เออแปลกดี'},
 {'message': 'ราคา 150 บาทมันเกินไปนะ รับไม่ไหวว'},
 {'message': 'ทอดมันหัวปลีกรอบอร่อยต้องเบิ้ล'}]

In [152]:
for i in tqdm(range(len(results_1000)), desc="Extracting sentiment"):
   result = sentiment_extract.batch(sentiment_inputs[i*3:(i*3)+3])
   sentiment_list = [d['sentiment'] for d in result]
   results_1000[i]['sentiment'] = sentiment_list

Extracting sentiment: 100%|██████████| 100/100 [04:30<00:00,  2.70s/it]


In [153]:
results_1000[0:2]

[{'review': 'ร้านอาหารใหญ่มากกกกกกก \nเลี้ยวเข้ามาเจอห้องน้ำก่อนเลย เออแปลกดี \nห้องทานหลักๆอยู่ชั้น 2 มีกาแฟ น้ำผึ้ง ซึ่งก็แค่เอาน้ำผึ้งมาราด แพงเวอร์ อย่าสั่งเลย \nลาบไข่ต้ม ไข่มันคาวอะ เลยไม่ประทับใจเท่าไหร่\nทอดมันหัวปลีกรอบอร่อยต้องเบิ้ล \nพะแนงห่อไข่อร่อยดี เห้ยแต่ราคา 150บาทมันเกินไปนะ รับไม่ไหวว\nเลิกกินแล้วมีขนมหวานให้กินฟรีเล็กน้อย )ขนมไทย) \n\nคงไม่ไปซ้ำ แพงเกิน ',
  'text': {'keywords': 'ราคา, บรรยากาศ, ไข่ต้ม'},
  'contexts': ['ร้านอาหารใหญ่มากกกกกกก',
   'ร้านอาหารใหญ่มากกกกกกก เลี้ยวเข้ามาเจอห้องน้ำก่อนเลย เออแปลกดี',
   'ห้องทานอาหารอยู่ชั้น 2 มีกาแฟ น้ำผึ้ง ซึ่งก็แค่เอาน้ำผึ้งมาราด แพงเวอร์ อย่าสั่งเลย'],
  'sentiment': ['neutral', 'neutral', 'negative']},
 {'review': 'วันนี้ได้มีโอกาสไปนั่งซดกาแฟที่ร้านวาวี แถวๆอารีย์\n\nก็ได้ยินมาบ่อยๆละ จากหลายที่ แต่ที่บ่อยสุดก็จาก Twitter ว่ากาแฟที่นี่อร่อยมากกกกกกก\nเรียกว่าระดับแฟนๆ Starbuck ยังต้องเหลียวมามอง ก็ดองมานานจนถึงวันนี้โอกาสเหมาะ ไป\npresent งานที่ตึก Software Park เสร็จก็เลยมาหาไรกินแถวนี้พอดี\nจัดไป.....\n\n\nบรรยา

In [154]:
import pandas as pd

df = pd.DataFrame([
    {
        'review': r['review'],
        **{f'keyword{i+1}': k.strip() for i, k in enumerate(r['text']['keywords'])},
        **{f'context{i+1}_keyword': c for i, c in enumerate(r['contexts'])},
        **{f'sentiment{i+1}': s for i, s in enumerate(r['sentiment'])}
    }
    for r in results_1000
])

In [155]:
df.to_csv('results.csv', index=False, encoding='utf-8-sig')